# Préparation du Dataset Éco-score via l'API Open Food Facts

Ce notebook permet de récupérer un jeu de données de produits alimentaires via l'API d'Open Food Facts dans le but de construire un modèle pour prédire l'éco-score (ecoscore_score ou ecoscore_grade) d'un produit, à partir de ses caractéristiques intrinsèques (catégorie, origine, emballage, labels, etc.).

## 1. Importation des bibliothèques nécessaires

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import time

try:
    from tqdm import tqdm  # Version standard pour éviter l'erreur liée à ipywidgets dans les notebooks
except ImportError:
    # Fallback si tqdm n'est pas installé
    def tqdm(iterator, *args, **kwargs):
        return iterator

pd.set_option('display.max_columns', 50)

## 2. Définition de la fonction de récupération via l'API

Pour constituer notre jeu de données, nous allons récupérer les produits les plus scannés pour lesquels l'éco-score est potentiellement disponible. La documentation indique que nous pouvons interroger `world.openfoodfacts.org/cgi/search.pl` par blocs.

In [ ]:
def fetch_openfoodfacts_ecoscore_data(pages=20, page_size=250):
    """
    Récupère un certain nombre de pages de l'API Open Food Facts avec gestion avancée des erreurs 503/Timeout.
    Renvoie un DataFrame pandas contenant les caractéristiques utiles à la prédiction de l'éco-score.
    """
    base_url = "https://world.openfoodfacts.org/cgi/search.pl"
    all_products = []
    
    # Entête User-Agent requis par l'API pour éviter les erreurs 403 / 503
    headers = {
        "User-Agent": "EcoScoreMLProject/1.1 - Python Script (Retry Enabled)"
    }
    
    # Configuration de la session avec Retry (pour éviter de bloquer à cause de 503)
    session = requests.Session()
    retries = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))
    session.mount('http://', HTTPAdapter(max_retries=retries))
    
    # Champs intéressants pour le calcul de l'éco-score : labels, origines, emballages, catégories
    champs = (
        "code,product_name,nutriscore_grade,ecoscore_score,ecoscore_grade,"
        "categories,labels,packaging,origins,ingredients_text,ingredients_analysis_tags"
    )
    
    for page in tqdm(range(1, pages + 1), desc="Téléchargement des pages"):
        params = {
            "action": "process",
            "sort_by": "unique_scans_n",  # Produits les plus populaires d'abord
            "page_size": page_size,
            "page": page,
            "json": 1,
            "fields": champs
        }
        
        try:
            # Requête vers l'API
            reponse = session.get(base_url, params=params, headers=headers, timeout=30)
            reponse.raise_for_status()
            donnees = reponse.json()
            
            if 'products' in donnees and len(donnees['products']) > 0:
                all_products.extend(donnees['products'])
            else:
                print(f"Plus de produits trouvés à partir de la page {page}.")
                break
                
        except Exception as e:
            print(f"\nErreur persistante rencontrée à la page {page} : {e}")
            break # On arrête la boucle pour renvoyer au moins les produits déjà récupérés
            
        # Temporisation selon les bonnes pratiques de l'API Open Food Facts (max 1-2 req/sec)
        time.sleep(1)
        
    print(f"\nRécupération terminée : {len(all_products)} produits téléchargés.")
    return pd.DataFrame(all_products)

## 3. Extraction des données

Nous allons récupérer les 5 premières pages (soit 5 000 produits bruts pour commencer).

In [ ]:
# Cette étape peut prendre un peu plus de temps. 
# En baissant page_size à 250 (au lieu de 1000), cela empêche les erreurs 503 (Timeouts) de l'API.
# Avec 20 pages de 250, nous obtenons toujours 5000 produits (comme 5x1000).
df_brut = fetch_openfoodfacts_ecoscore_data(pages=20, page_size=250)
print(f"Taille du dataframe brut : {df_brut.shape}")

## 4. Nettoyage et préparation du dataset cible

L'objectif est de prédire l'éco-score. Nous devons donc exclure les produits pour lesquels nous n'avons pas d'éco-score.

In [ ]:
# Visualisation d'un échantillon
if not df_brut.empty:
    display(df_brut.head())
else:
    print("Le dataframe est vide, l'API n'a rien renvoyé.")

In [ ]:
if not df_brut.empty and 'ecoscore_grade' in df_brut.columns:
    # On supprime les lignes où l'ecoscore n'est pas renseigné ou catégorisé comme 'unknown'
    df_eco = df_brut.dropna(subset=['ecoscore_grade', 'ecoscore_score']).copy()
    df_eco = df_eco[df_eco['ecoscore_grade'] != 'unknown']
    df_eco = df_eco[df_eco['ecoscore_grade'].notnull() & (df_eco['ecoscore_grade'] != 'not-applicable')]

    print(f"Nombre de produits exploitables (avec éco-score connu) : {len(df_eco)}")
else:
    print("Pas de données suffisantes pour le nettoyage.")
    df_eco = pd.DataFrame()

## 5. Prétraitement des variables textuelles qui serviront de Features

In [ ]:
# Remplacer les valeurs manquantes par des chaînes vides pour l'analyse NLP
colonnes_features = ['categories', 'labels', 'packaging', 'origins', 'ingredients_text']

if not df_eco.empty:
    for col in colonnes_features:
        if col in df_eco.columns:
            df_eco[col] = df_eco[col].fillna('')
            # Mettre en minuscules pour limiter la variabilité due à la casse
            df_eco[col] = df_eco[col].astype(str).str.lower()

In [ ]:
# Extraction des valeurs de distribution de l'Eco-score
if not df_eco.empty:
    df_eco['ecoscore_grade'].value_counts().sort_index().plot(kind='bar', title='Répartition des Éco-scores dans le dataset')

## 6. Sauvegarde du DataSet nettoyé

Nous pouvons maintenant exporter le DataFrame sous format CSV pour l'utiliser lors de la phase d'entraînement (Machine Learning / prédiction de l'éco-score).

In [ ]:
if not df_eco.empty:
    chemin_sortie = "dataset_openfoodfacts_ecoscore.csv"
    df_eco.to_csv(chemin_sortie, index=False)
    print(f"Dataset final enregistré vers : {chemin_sortie}")
else:
    print("Aucun dataset généré à cause de l'erreur d'API.")